![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/llama.cpp/DocumentTranslator.ipynb)

# Translate Documents with DocumentTranslator and GGUF Models

`DocumentTranslator` translates complete documents with GGUF models in Spark NLP. With compatible multilingual pretrained models, it supports translation across more than 180 languages. You only need to set the source and target languages.

In this notebook, we will use a quantized [Tower+ 9B](https://huggingface.co/Unbabel/Tower-Plus-9B) model to translate English documents into French.

## Download the GGUF Model and Test Documents

Let's download the `Q2_K` quantized Tower+ 9B model from [Hugging Face](https://huggingface.co/mradermacher/Tower-Plus-9B-GGUF) and the public test documents from the Spark NLP GitHub repository.


In [1]:
EXPORT_PATH = "/content/Tower-Plus-9B.Q2_K.gguf"
TEST_FILES_DIR = "/content/document-translator-test-files"

!mkdir -p {TEST_FILES_DIR}

!wget -q --show-progress "https://huggingface.co/mradermacher/Tower-Plus-9B-GGUF/resolve/main/Tower-Plus-9B.Q2_K.gguf?download=true" -O {EXPORT_PATH}
!wget -q "https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/SPARKNLP-1251-Create-end-to-end-DocumentTranslator/src/test/resources/reader/txt/long-text.txt" -O {TEST_FILES_DIR}/long-text.txt

!ls -lh {EXPORT_PATH} {TEST_FILES_DIR}

/content/Tower-Plus 100%[===================>]   3.54G  40.1MB/s    in 99s     
-rw-r--r-- 1 root root 3.6G Jul  2 14:07 /content/Tower-Plus-9B.Q2_K.gguf

/content/document-translator-test-files:
total 20K
-rw-r--r-- 1 root root 17K Jul  2 14:07 long-text.txt


## Import DocumentTranslator in Spark NLP

- Let's install and set up Spark NLP if this notebook is running in Google Colab.
- This part is pretty easy via our simple script.


In [2]:
# Only execute this if you are on Google Colab
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash


Installing PySpark 3.4.4 and Spark NLP 6.4.2
setup Colab for PySpark 3.4.4 and Spark NLP 6.4.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.4 which is incompatible.


Let's start Spark with Spark NLP included via our simple `start()` function.


In [ ]:
import sparknlp

# Let's start Spark with Spark NLP with GPU enabled. If you don't have GPUs available, remove this parameter.
spark = sparknlp.start(gpu=True)

print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)


Spark NLP version: 6.4.1
Apache Spark version: 3.4.4


Let's load the downloaded GGUF model with `DocumentTranslator.loadSavedModel()` and set the source language to English and the target language to French.



In [5]:
from sparknlp.annotator import DocumentTranslator

document_translator = (
    DocumentTranslator.loadSavedModel(EXPORT_PATH, spark)
    .setContentPath(TEST_FILES_DIR)
    .setOutputCol("translation")
    .setSrcLang("English")
    .setTgtLang("French")
    .setMinSentenceLength(800)
    .setMaxSentenceLength(1500)
    .setBatchSize(4)
    .setNPredict(512)
    .setNCtx(8192)
    .setTemperature(0.0)
    .setNGpuLayers(99)
)


## Translate the Test Documents

Now let's add `DocumentTranslator` to a Spark ML pipeline and trnssate the document.

In [6]:
from pyspark.ml import Pipeline

pipeline = Pipeline().setStages([document_translator])

empty_data = spark.createDataFrame([[""]]).toDF("text")
result = pipeline.fit(empty_data).transform(empty_data)

result.select("fileName", "translation.result").show(truncate=False)


+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

That's it! You can now translate complete documents with GGUF models in Spark NLP.
